# Chapter 10 explore: Traceable Outputs and Hallucination Mitigation

Interactive companion to `code/chapter_10/traceable_outputs.py`. Scores each of Chapter 9's real answers against the individual source chunks it was shown, and keeps only the sources an answer is actually faithful to. Needs Chapter 8's checkpoint to exist first (`python code/chapter_08/finetune_at_scale.py`, ~30 min).

In [1]:
import sys
sys.path.insert(0, "../code/chapter_01")
sys.path.insert(0, "../code/chapter_02")
sys.path.insert(0, "../code/chapter_06")
sys.path.insert(0, "../code/chapter_07")
sys.path.insert(0, "../code/chapter_09")
sys.path.insert(0, "../code/chapter_10")

from load_local_model import MODEL_NAME, load_model_and_tokenizer
from hybrid_rag_finetune import INSTRUCTION, TEST_CASES, build_retrieval_corpus, build_bm25_index, latest_checkpoint
from traceable_outputs import answer_with_traceable_sources, faithfulness_score
from peft import PeftModel

corpus = build_retrieval_corpus()
bm25 = build_bm25_index(corpus)
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
lora_model = PeftModel.from_pretrained(model, latest_checkpoint())
print("Ready.")

`torch_dtype` is deprecated! Use `dtype` instead!


Ready.


Try your own answer/source pair -- see how the score reacts to a paraphrase versus a real mismatch.

In [2]:
answer = "Test choke manifold at 5000psi"  # try your own
source = "test choke manifold 5000 psi"  # try your own
print(faithfulness_score(answer, source))

0.75


Run the full traceable-sources check across all 4 of Chapter 9's test cases.

In [3]:
for label, input_context, query, target_report in TEST_CASES:
    result = answer_with_traceable_sources(lora_model, tokenizer, INSTRUCTION, input_context, query, corpus, bm25)
    print(f"{label}: {result['answer']}")
    print(f"  grounded: {result['grounded']}")
    for chunk in result["retrieved"]:
        mark = "verified" if chunk["faithfulness"] >= 0.5 else "not used"
        print(f"    [{mark:>8}] Report #{chunk['report_num']} (faithfulness {chunk['faithfulness']:.2f})")
    print()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Report #37 (held-out): Trip out of hole with BHA #18. Stop at 5,800' and circulate to cool hole and tools.
  grounded: True
    [verified] Report #37 (faithfulness 1.00)
    [verified] Report #28 (faithfulness 0.78)
    [not used] Report #39 (faithfulness 0.44)



Report #38 (stuck pipe): Production Drilling Trips Trip in hole with drill pipe from 10,490' to surface
  grounded: False
    [not used] Report #38 (faithfulness 0.44)
    [not used] Report #60 (faithfulness 0.22)
    [not used] Report #61 (faithfulness 0.11)



Report #21 (step rate test): Test choke manifold at 5000psi
  grounded: True
    [verified] Report #27 (faithfulness 0.75)
    [not used] Report #21 (faithfulness 0.25)
    [not used] Report #17 (faithfulness 0.25)



Report #49 (fishing): Trip out of hole with fishing bha
  grounded: True
    [verified] Report #49 (faithfulness 0.80)
    [not used] Report #49 (faithfulness 0.40)
    [not used] Report #50 (faithfulness 0.20)

